# PyMuPDF GESB Table Experiment Walkthrough

This notebook runs the PyMuPDF GESB SAFF table extraction experiment step by step.

It is designed for inspection rather than production parsing. The cells expose the intermediate state used by `pymupdf_table_experiment.py`: page words, detected layout, row starts, PyMuPDF `find_tables()` output, reconstructed rows, and quality checks.

## 1. Setup

Run this notebook from anywhere inside the repository. The setup cell locates the repo root and adds `src` to `sys.path` so the local parser package can be imported without installing it first.

In [1]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find repository root from current working directory")


REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("repo root:", REPO_ROOT)
print("src root:", SRC_ROOT)

repo root: C:\Users\hy120\Downloads\AI project\ContextBridge
src root: C:\Users\hy120\Downloads\AI project\ContextBridge\src


## 2. Import The Experiment Module

The notebook imports the experiment module directly and reuses its helper functions. This keeps the notebook aligned with the script implementation.

In [2]:
import fitz

from contextbridge_parser.parsers.pdf.sources.gesb import pymupdf_table_experiment as exp

SOURCE_PATH = (REPO_ROOT / exp.DEFAULT_SOURCE_PATH).resolve()
OUTPUT_PATH = (REPO_ROOT / exp.DEFAULT_OUTPUT_PATH).resolve()

print("PyMuPDF version:", fitz.VersionBind)
print("source exists:", SOURCE_PATH.exists(), SOURCE_PATH)
print("output path:", OUTPUT_PATH)
print("page range:", exp.TABLE_PAGE_START, exp.TABLE_PAGE_END)

PyMuPDF version: 1.27.2.3
source exists: True C:\Users\hy120\Downloads\AI project\ContextBridge\knowledge_base\sources\structured-business-file-formats\saff\specifications\gesb\superstream-payroll-data-specification.pdf
output path: C:\Users\hy120\Downloads\AI project\ContextBridge\knowledge_base\processed\structured-business-file-formats\saff\specifications\gesb\superstream-payroll-data-specification.pymupdf-tables.json
page range: 10 27


## 3. Inspect One Page

Start with page 10 because it contains clean examples and was used during the initial experiment notes.

In [6]:
doc = fitz.open(SOURCE_PATH)
page_number = 10
page = doc[page_number - 1]

words = exp._page_words(page)

print("page:", page_number)
print("page size:", page.rect)
print("word count:", len(words))
pprint(words[:20])

page: 10
page size: Rect(0.0, 0.0, 842.0399780273438, 595.3200073242188)
word count: 158
[{'block_no': 0,
  'line_no': 0,
  'text': 'Government',
  'word_no': 0,
  'x0': 45.119998931884766,
  'x1': 94.72799682617188,
  'y0': 35.65498733520508,
  'y1': 45.68998718261719},
 {'block_no': 0,
  'line_no': 0,
  'text': 'Employees',
  'word_no': 1,
  'x0': 97.25699615478516,
  'x1': 141.67198181152344,
  'y0': 35.65498733520508,
  'y1': 45.68998718261719},
 {'block_no': 0,
  'line_no': 0,
  'text': 'Superannuation',
  'word_no': 2,
  'x0': 144.11097717285156,
  'x1': 207.64193725585938,
  'y0': 35.65498733520508,
  'y1': 45.68998718261719},
 {'block_no': 0,
  'line_no': 0,
  'text': 'Board',
  'word_no': 3,
  'x0': 210.17092895507812,
  'x1': 234.13790893554688,
  'y0': 35.65498733520508,
  'y1': 45.68998718261719},
 {'block_no': 0,
  'line_no': 0,
  'text': '(GESB)',
  'word_no': 4,
  'x0': 236.89999389648438,
  'x1': 267.8689880371094,
  'y0': 35.65498733520508,
  'y1': 45.68998718261719},


## 4. Detect Column Layout

This cell shows whether the layout came from detected header words or from the hard-coded fallback ranges.

In [7]:
layout = exp._detect_table_layout(words)
print(json.dumps(layout, indent=2))

{
  "x_ranges": {
    "column_number": [
      44.400001525878906,
      92.46399688720703
    ],
    "field_name": [
      92.46399688720703,
      205.6199951171875
    ],
    "description": [
      205.6199951171875,
      347.489990234375
    ],
    "requirements_label": [
      347.489990234375,
      410.489990234375
    ],
    "requirements_value": [
      410.489990234375,
      510.3499755859375
    ],
    "required_by_gesb": [
      513.3499755859375,
      565.9099731445312
    ],
    "mig_reference": [
      565.9099731445312,
      619.8200073242188
    ],
    "des_reference": [
      619.8200073242188,
      701.8200073242188
    ]
  },
  "table_data_y_min": 135.91500091552734,
  "source": "header_words"
}


## 5. Find Row Starts

Row starts are detected from numeric words in the `column_number` x range. The current rule also requires nearby field words and right-side or requirements words.

In [8]:
row_starts = exp._find_row_starts(words, layout)

print("row start count:", len(row_starts))
for start in row_starts:
    print(
        f"column={start['text']:>3} "
        f"x0={start['x0']:.3f} y0={start['y0']:.3f} "
        f"text={start['text']!r}"
    )

row start count: 6
column=  1 x0=56.400 y0=154.624 text='1'
column=  2 x0=56.400 y0=204.814 text='2'
column=  3 x0=56.400 y0=254.614 text='3'
column=  4 x0=56.400 y0=310.554 text='4'
column=  5 x0=56.400 y0=360.474 text='5'
column=  6 x0=56.400 y0=410.514 text='6'


## 6. Compare PyMuPDF `find_tables()`

`find_tables()` is useful as a diagnostic signal, but the experiment does not use it as the primary extraction path because it misses some SAFF rows.

In [9]:
finder_tables = exp._find_tables_summary(page)

print("find_tables count:", len(finder_tables))
for table in finder_tables:
    print("---")
    print("table_index:", table["table_index"])
    print("bbox:", table["bbox"])
    print("rows:", table["row_count"], "cols:", table["col_count"])
    pprint(table["preview_rows"])

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
find_tables count: 3
---
table_index: 0
bbox: [51.0, 154.362, 673.294, 204.669]
rows: 5 cols: 9
[['1', 'Version', 'Heading text', '', 'Mandatory:', 'Yes', 'No', 'N/A', 'N/A'],
 ['', '', '', '', 'Data Type:', 'String', '', '', ''],
 [None, None, None, None, 'Length:', '7', None, None, None],
 [None, None, None, None, 'Value(s):', 'VERSION', None, None, None],
 [None, None, None, None, '', '', None, None, None]]
---
table_index: 1
bbox: [51.0, 254.716, 673.294, 310.382]
rows: 5 cols: 9
[['3',
  'Negatives Supported',
  'Heading text',
  '',
  'Mandatory:',
  'Yes',
  'No',
  'N/A',
  'N/A'],
 ['', '', '', '', 'Data Type:', 'String', '', '', ''],
 [None, None, None, None, 'Length:', '19', None, None, None],
 [None, None, None, None, 'Value(s):', 'NEGATIVES', None, None, None],
 [None, None, None, None, '', 'SUPPORTED', None, None, None]]
---
table_index: 2
bbox: [51.0, 360.576, 673.294, 410.369]
rows: 5

## 7. Reconstruct Rows From Words

This is the main extraction path. Words are grouped between one row start and the next row start, then assigned to cells by x range.

In [10]:
page_rows = exp._extract_rows_from_words(page_number, words, row_starts, layout)

print("reconstructed row count:", len(page_rows))
for row in page_rows:
    print("---")
    pprint(
        {
            "page_number": row["page_number"],
            "column_number": row["column_number"],
            "field_name": row["field_name"],
            "description": row["description"],
            "requirements": row["requirements"],
            "required_by_gesb": row["required_by_gesb"],
            "mig_reference": row["mig_reference"],
            "des_reference": row["des_reference"],
        }
    )

reconstructed row count: 6
---
{'column_number': 1,
 'des_reference': 'N/A',
 'description': 'Heading text',
 'field_name': 'Version',
 'mig_reference': 'N/A',
 'page_number': 10,
 'required_by_gesb': 'No',
 'requirements': {'data_type': 'String',
                  'format': None,
                  'length': '7',
                  'mandatory': 'Yes',
                  'notes': None,
                  'values': 'VERSION'}}
---
{'column_number': 2,
 'des_reference': 'N/A',
 'description': 'Version number',
 'field_name': 'Version data',
 'mig_reference': 'N/A',
 'page_number': 10,
 'required_by_gesb': 'No',
 'requirements': {'data_type': 'String',
                  'format': None,
                  'length': '3',
                  'mandatory': 'Yes',
                  'notes': None,
                  'values': '1.0'}}
---
{'column_number': 3,
 'des_reference': 'N/A',
 'description': 'Heading text',
 'field_name': 'Negatives Supported',
 'mig_reference': 'N/A',
 'page_number': 10,
 'requi

## 8. Run The Full Experiment

This runs the same high-level function used by the script. It keeps the result in memory so it can be inspected before writing JSON.

In [ ]:
result = exp.extract_pymupdf_tables(SOURCE_PATH)
rows = result["rows"]
pages = result["pages"]

print("pages:", len(pages))
print("rows:", len(rows))

for page_info in pages:
    print(
        page_info["page_number"],
        "rows=", page_info["word_row_count"],
        "starts=", page_info["word_row_start_count"],
        "find_tables=", len(page_info["pymupdf_find_tables"]),
        "layout=", page_info["detected_layout"]["source"],
    )

## 9. Quality Check

This cell checks the known quality risks: missing rows, duplicate numbers, polluted headers, invalid `required_by_gesb` values, and empty key fields.

In [ ]:
EXPECTED_COLUMN_NUMBERS = set(range(1, 134))
ALLOWED_REQUIRED_BY_GESB = {
    "Yes",
    "No",
    "No - Ignored by GESB",
    "No – Ignored by GESB",
}
HEADER_MARKERS = (
    "Field Name",
    "Description",
    "Requirements",
    "Required by GESB?",
    "MIG 2.0 Reference",
    "DES Spec 5.8 Reference",
)
SECTION_MARKERS = ("Section:", "LINE ID", "11.4.")


def contains_any(value: str | None, markers: tuple[str, ...]) -> bool:
    return bool(value and any(marker in value for marker in markers))


def quality_report(rows: list[dict]) -> dict:
    numbers = [row.get("column_number") for row in rows]
    number_counts = Counter(numbers)
    present_numbers = {number for number in numbers if number is not None}

    key_fields = (
        "column_number",
        "field_name",
        "description",
        "requirements_text",
        "required_by_gesb",
        "mig_reference",
        "des_reference",
    )

    polluted_rows = []
    invalid_required_rows = []
    for row in rows:
        searchable = " ".join(str(row.get(field) or "") for field in key_fields)
        if contains_any(searchable, HEADER_MARKERS) or contains_any(searchable, SECTION_MARKERS):
            polluted_rows.append(row)
        if row.get("required_by_gesb") not in ALLOWED_REQUIRED_BY_GESB:
            invalid_required_rows.append(row)

    return {
        "row_count": len(rows),
        "missing_column_numbers": sorted(EXPECTED_COLUMN_NUMBERS - present_numbers),
        "duplicate_column_numbers": {
            number: count for number, count in sorted(number_counts.items()) if count > 1
        },
        "empty_field_counts": {
            field: sum(1 for row in rows if row.get(field) in (None, ""))
            for field in key_fields
        },
        "invalid_required_by_gesb": [
            {
                "page_number": row.get("page_number"),
                "column_number": row.get("column_number"),
                "field_name": row.get("field_name"),
                "required_by_gesb": row.get("required_by_gesb"),
            }
            for row in invalid_required_rows
        ],
        "polluted_rows": [
            {
                "page_number": row.get("page_number"),
                "column_number": row.get("column_number"),
                "field_name": row.get("field_name"),
                "required_by_gesb": row.get("required_by_gesb"),
                "mig_reference": row.get("mig_reference"),
                "des_reference": row.get("des_reference"),
            }
            for row in polluted_rows
        ],
    }


report = quality_report(rows)
pprint(report)

## 10. Inspect Missing Or Polluted Rows

Use this cell when the quality report identifies a missing or polluted case. Change `TARGET_COLUMN_NUMBERS` to focus on specific columns.

In [ ]:
TARGET_COLUMN_NUMBERS = report["missing_column_numbers"][:]

print("missing column numbers:", TARGET_COLUMN_NUMBERS)
print("nearby extracted rows:")

for target in TARGET_COLUMN_NUMBERS:
    nearby = [
        row for row in rows
        if row.get("column_number") is not None
        and target - 2 <= row["column_number"] <= target + 2
    ]
    print("\n=== target", target, "===")
    for row in nearby:
        print(
            row["page_number"],
            row["column_number"],
            row["field_name"],
            "| required_by_gesb=", row["required_by_gesb"],
            "| mig=", row["mig_reference"],
            "| des=", row["des_reference"],
        )

## 11. Write Experiment Output

Run this cell when you want to regenerate the JSON output file from the notebook. The current script writer should be updated to write UTF-8 explicitly before this output is used by downstream tooling.

In [ ]:
# Uncomment when you want to write the current experiment output.
# written_path = exp.write_experiment_output(SOURCE_PATH, OUTPUT_PATH)
# print("wrote:", written_path)